In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path(
    "../data/raw/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"
)

df = pd.read_csv(RAW_FILE)

print("Raw shape:", df.shape)

Raw shape: (1048575, 80)


In [2]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

print(
    "Timestamp parsing failures:",
    df["Timestamp"].isna().sum()
)

Timestamp parsing failures: 0


In [3]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Duplicates removed:", before - after)

Rows before duplicate removal: 1048575
Rows after duplicate removal: 1031018
Duplicates removed: 17557


In [4]:
df = df.sort_values(
    "Timestamp"
).reset_index(drop=True)

print("✅ Dataset sorted chronologically")

print("First:", df["Timestamp"].min())
print("Last :", df["Timestamp"].max())

✅ Dataset sorted chronologically
First: 2018-02-21 01:55:46
Last : 2018-02-21 10:43:21


In [5]:
df["Target"] = (
    ~df["Label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("benign")
).astype(int)

print(
    df[["Label", "Target"]]
    .drop_duplicates()
    .sort_values("Target")
)

                        Label  Target
0                      Benign       0
69           DDOS attack-HOIC       1
1027609  DDOS attack-LOIC-UDP       1


In [6]:
CONSTANT_FEATURES = [
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg"
]

DROP_COLUMNS = [
    "Timestamp",
    "Label"
]

df_model = df.drop(
    columns=DROP_COLUMNS + CONSTANT_FEATURES,
    errors="ignore"
).copy()

print("Model-ready shape:", df_model.shape)

Model-ready shape: (1031018, 69)


In [7]:
n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (721712, 69)
Validation: (154653, 69)
Test: (154653, 69)


In [8]:
print("TRAIN")
print(train_df["Target"].value_counts())

print("\nVALIDATION")
print(val_df["Target"].value_counts())

print("\nTEST")
print(test_df["Target"].value_counts())

TRAIN
Target
1    520348
0    201364
Name: count, dtype: int64

VALIDATION
Target
0    78669
1    75984
Name: count, dtype: int64

TEST
Target
0    80794
1    73859
Name: count, dtype: int64


In [9]:
for name, part in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    
    print(f"\n{name}")

    percentages = (
        part["Target"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )

    print(percentages)


TRAIN
Target
0    27.90088
1    72.09912
Name: proportion, dtype: float64

VALIDATION
Target
0    50.868072
1    49.131928
Name: proportion, dtype: float64

TEST
Target
0    52.242116
1    47.757884
Name: proportion, dtype: float64


In [10]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = PROCESSED_DIR / "train_dos.csv"
VAL_FILE = PROCESSED_DIR / "validation_dos.csv"
TEST_FILE = PROCESSED_DIR / "test_dos.csv"

train_df.to_csv(TRAIN_FILE, index=False)
val_df.to_csv(VAL_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print("✅ Official dataset splits saved")
print("Train:", TRAIN_FILE, train_df.shape)
print("Validation:", VAL_FILE, val_df.shape)
print("Test:", TEST_FILE, test_df.shape)

✅ Official dataset splits saved
Train: ..\data\processed\train_dos.csv (721712, 69)
Validation: ..\data\processed\validation_dos.csv (154653, 69)
Test: ..\data\processed\test_dos.csv (154653, 69)
